In [144]:
import os
import math
import random
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader, Subset
from torch_geometric.nn import GATv2Conv
from torch_geometric.data import Data, Batch
from torch_geometric.loader import DataLoader as GraphDataLoader

from tqdm import tqdm
from sklearn.metrics import precision_score, recall_score, f1_score, fbeta_score
from transformers import set_seed, default_data_collator

from tsfm_public.models.tspulse import TSPulseForReconstruction
from tsfm_public.toolkit.time_series_preprocessor import (
    TimeSeriesPreprocessor,
    get_datasets,
)


set_seed(42)
random.seed(42)
np.random.seed(42)
torch.manual_seed(42)

CONTEXT_LENGTH = 512

device = torch.device("cuda" if torch.cuda.is_available() else "mps" if torch.backends.mps.is_available() else "cpu")

In [145]:
path = "/Users/darenpalmer/Desktop/UCL/CS/fyp.nosync/data/carOBD/obdiidata" 
time_col = 'ENGINE_RUN_TINE ()'

target_columns = [
    'COOLANT_TEMPERATURE ()',
    'ENGINE_RPM ()',
    'VEHICLE_SPEED ()',
    'THROTTLE ()',
    'ENGINE_LOAD ()',
    'INTAKE_MANIFOLD_PRESSURE ()',
]

df_list = []
for file in os.listdir(path):
    if file.endswith('.csv'):
        df = pd.read_csv(f'{path}/{file}', index_col=False)
        df['filename'] = file
        df_list.append(df)




def mean_fill_missing_timestamps_and_remove_duplicates(df: pd.DataFrame) -> pd.DataFrame:
    df = df.groupby(time_col, as_index=False).mean(numeric_only=True)
    return df

dfs = []
for drive_idx, df in enumerate(df_list):
    df_clean = mean_fill_missing_timestamps_and_remove_duplicates(df)
    
    missing_cols = [col for col in target_columns if col not in df_clean.columns]
    if missing_cols or time_col not in df_clean.columns:
        continue
    
    df_clean['drive_id'] = drive_idx
    required_cols = [time_col, 'drive_id'] + target_columns
    df_clean = df_clean[required_cols].copy()
    
    if not pd.api.types.is_numeric_dtype(df_clean[time_col]):
        df_clean[time_col] = pd.to_numeric(df_clean[time_col], errors='coerce')
    
    df_clean = df_clean.dropna(subset=[time_col] + target_columns)
    
    if len(df_clean) > 0:
        dfs.append(df_clean)

data = pd.concat(dfs, ignore_index=True)


In [146]:
data['drive_id'] = data['drive_id'].astype(str)
all_drive_ids = data['drive_id'].unique()

train_ids = all_drive_ids[:int(0.7 * len(all_drive_ids))]
val_ids = all_drive_ids[int(0.7 * len(all_drive_ids)):int(0.85 * len(all_drive_ids))]
test_ids = all_drive_ids[int(0.85 * len(all_drive_ids)):]

train_data = data[data['drive_id'].isin(train_ids)].copy()
val_data = data[data['drive_id'].isin(val_ids)].copy()
test_data = data[data['drive_id'].isin(test_ids)].copy()


In [147]:
from faults import create_realistic_fault_data

In [148]:
tsp = TimeSeriesPreprocessor(
    timestamp_column=time_col,
    id_columns=['drive_id'],
    target_columns=target_columns,
    control_columns=[],
    context_length=CONTEXT_LENGTH,
    prediction_length=0,
    scaling=True,
    scaling_id_columns=[],
    scaling_type="robust"
)

tsp = tsp.train(train_data)

tspulse_model = TSPulseForReconstruction.from_pretrained("./tspulse_obd_finetuned")
tspulse_model.to(device)
tspulse_model.eval()


TSPulseForReconstruction(
  (loss): MSELoss()
  (backbone): TSPulseModel(
    (encoder_block): TSPulseBlock(
      (mixers): ModuleList(
        (0-7): 8 x TSPulseLayer(
          (patch_mixer): PatchMixerBlock(
            (norm): TSPulseNormLayer(
              (norm): LayerNorm((24,), eps=1e-05, elementwise_affine=True)
            )
            (mlp): TSPulseMLP(
              (fc1): Linear(in_features=138, out_features=276, bias=True)
              (dropout1): Dropout(p=0.2, inplace=False)
              (fc2): Linear(in_features=276, out_features=138, bias=True)
              (dropout2): Dropout(p=0.2, inplace=False)
            )
            (gating_block): TSPulseGatedAttention(
              (attn_layer): Linear(in_features=138, out_features=138, bias=True)
              (attn_activation_layer): Softmax(dim=-1)
            )
          )
          (feature_mixer): FeatureMixerBlock(
            (norm): TSPulseNormLayer(
              (norm): LayerNorm((24,), eps=1e-05, elementwi

In [149]:
train_dset, _, _ = get_datasets(tsp, train_data, split_config={"train": 1.0, "test": 0.0})
val_dset, _, _ = get_datasets(tsp, val_data, split_config={"train": 1.0, "test": 0.0})
test_dset, _, _ = get_datasets(tsp, test_data, split_config={"train": 1.0, "test": 0.0})

In [150]:
def extract_windows_from_dataset(dataset):
    windows = []
    for i in range(len(dataset)):
        sample = dataset[i]
        window = sample['past_values']
        windows.append(window.numpy())
    return windows

train_windows_clean = extract_windows_from_dataset(train_dset)
val_windows_clean = extract_windows_from_dataset(val_dset)
test_windows_clean = extract_windows_from_dataset(test_dset)

In [151]:
def inject_faults_into_windows(windows, fault_percentage=0.05, random_state=42):
    if random_state is not None:
        np.random.seed(random_state)
    
    faulty_windows = []
    all_labels = []
    
    for window in tqdm(windows, desc="Injecting faults"):
        faulty_window, window_labels, _ = create_realistic_fault_data(
            window,
            fault_percentage=fault_percentage,
            random_state=None
        )
        
        faulty_windows.append(faulty_window)
        all_labels.append(window_labels)
    
    point_labels = np.concatenate(all_labels)
    
    return faulty_windows, point_labels

train_windows_faulty, train_point_labels = inject_faults_into_windows(
    train_windows_clean, fault_percentage=0.05, random_state=42
)
val_windows_faulty, val_point_labels = inject_faults_into_windows(
    val_windows_clean, fault_percentage=0.15, random_state=43
)
test_windows_faulty, test_point_labels = inject_faults_into_windows(
    test_windows_clean, fault_percentage=0.20, random_state=44
)

Injecting faults: 100%|██████████| 2462/2462 [00:03<00:00, 629.30it/s]


In [ ]:
def create_timestep_graphs_vectorized(window_data, reconstruction, labels_512, device):
    """
    Vectorized version that processes all timesteps efficiently on the specified device.
    Works on both CPU and CUDA.
    """
    # Convert to tensors and move to device
    window_data = torch.tensor(window_data, dtype=torch.float32).to(device)
    reconstruction = torch.tensor(reconstruction, dtype=torch.float32).to(device)
    labels_512 = torch.tensor(labels_512, dtype=torch.float32)
    
    T, num_sensors = window_data.shape
    graphs = []
    
    # Adjust batch size based on device (smaller batches for CPU)
    timestep_batch_size = 128 if device.type == 'cuda' else 32
    
    # Pre-compute sliding windows for all timesteps at once
    window_size = 21  # 10 before + 1 current + 10 after
    
    # Create padded arrays for efficient slicing
    pad_size = 10
    padded_window = F.pad(window_data, (0, 0, pad_size, pad_size), mode='edge')
    padded_recon = F.pad(reconstruction, (0, 0, pad_size, pad_size), mode='edge')
    
    # Process all timesteps in batches
    for batch_start in range(0, T, timestep_batch_size):
        batch_end = min(batch_start + timestep_batch_size, T)
        batch_timesteps = batch_end - batch_start
        
        # Extract local windows for this batch [batch_timesteps, window_size, num_sensors]
        local_windows = torch.zeros(batch_timesteps, window_size, num_sensors, device=device)
        local_recons = torch.zeros(batch_timesteps, window_size, num_sensors, device=device)
        
        for i, t in enumerate(range(batch_start, batch_end)):
            t_start = max(0, t - 10)
            t_end = min(T, t + 11)
            actual_start = pad_size + t_start
            actual_end = pad_size + t_end
            actual_len = t_end - t_start
            local_windows[i, :actual_len] = padded_window[actual_start:actual_end]
            local_recons[i, :actual_len] = padded_recon[actual_start:actual_end]
        
        # Vectorized feature computation for all timesteps in batch
        # Shape: [batch_timesteps, num_sensors, window_size]
        local_windows_perm = local_windows.permute(0, 2, 1)
        local_recons_perm = local_recons.permute(0, 2, 1)
        
        # Statistical features [batch_timesteps, num_sensors]
        feat_mean = local_windows_perm.mean(dim=2)
        feat_std = local_windows_perm.std(dim=2)
        feat_min = local_windows_perm.min(dim=2)[0]
        feat_max = local_windows_perm.max(dim=2)[0]
        feat_variance = local_windows_perm.var(dim=2)
        
        # Reconstruction error features
        error = torch.abs(local_windows_perm - local_recons_perm)
        feat_recon_error_mean = error.mean(dim=2)
        feat_recon_error_std = error.std(dim=2)
        feat_recon_error_max = error.max(dim=2)[0]
        
        # Velocity and jumps (using diff)
        diff = torch.diff(local_windows_perm, dim=2)
        feat_velocity = diff.mean(dim=2)
        diff_std = diff.std(dim=2)
        threshold = 3 * diff_std + 1e-6
        feat_jumps = (torch.abs(diff) > threshold.unsqueeze(2)).float().mean(dim=2)
        
        # Stack features: [batch_timesteps, num_sensors, 10]
        node_features = torch.stack([
            feat_mean, feat_std, feat_min, feat_max,
            feat_recon_error_mean, feat_recon_error_std, feat_recon_error_max,
            feat_velocity, feat_jumps, feat_variance
        ], dim=2)
        
        # Normalize per timestep
        mean = node_features.mean(dim=1, keepdim=True)
        std = node_features.std(dim=1, keepdim=True) + 1e-6
        node_features_normalized = (node_features - mean) / std
        
        # Compute correlations for all timesteps in batch
        for i, t in enumerate(range(batch_start, batch_end)):
            t_start = max(0, t - 10)
            t_end = min(T, t + 11)
            local_window = window_data[t_start:t_end, :]  # [window_size, num_sensors]
            
            # Vectorized correlation computation on device
            local_window_centered = local_window - local_window.mean(dim=0, keepdim=True)
            std_dev = local_window_centered.std(dim=0, keepdim=True) + 1e-6
            local_window_norm = local_window_centered / std_dev
            
            # Compute correlation matrix
            corr_matrix = torch.mm(local_window_norm.t(), local_window_norm) / (local_window.shape[0] - 1)
            
            # Extract upper triangle (excluding diagonal)
            edge_index = []
            edge_attr = []
            for sensor_i in range(num_sensors):
                for sensor_j in range(sensor_i + 1, num_sensors):
                    corr = corr_matrix[sensor_i, sensor_j].item()
                    if not torch.isnan(torch.tensor(corr)) and abs(corr) > 0.1:
                        edge_index.extend([[sensor_i, sensor_j], [sensor_j, sensor_i]])
                        edge_attr.extend([[corr], [corr]])
            
            if len(edge_index) == 0:
                edge_index = [[s_i, s_j] for s_i in range(num_sensors) for s_j in range(num_sensors) if s_i != s_j]
                edge_attr = [[0.0] for _ in edge_index]
            
            edge_index_tensor = torch.tensor(edge_index, dtype=torch.long, device=device).t()
            edge_attr_tensor = torch.tensor(edge_attr, dtype=torch.float32, device=device)
            
            # Move to CPU only at the end for graph creation
            graph = Data(
                x=node_features_normalized[i].cpu(),
                edge_index=edge_index_tensor.cpu(),
                edge_attr=edge_attr_tensor.cpu(),
                y=labels_512[t:t+1],
                num_nodes=num_sensors
            )
            graphs.append(graph)
    
    return graphs


def create_graphs_from_faulty_windows_batched(windows_faulty, labels_per_window, tspulse_model, device, batch_size=None):
    """
    Process windows in batches to better utilize GPU/CPU.
    Automatically adjusts batch size based on device if not specified.
    """
    all_graphs = []
    
    # Auto-adjust batch size based on device
    if batch_size is None:
        if device.type == 'cuda':
            batch_size = 16  # Larger batches for GPU
        else:
            batch_size = 4   # Smaller batches for CPU/MPS
    
    tspulse_model.eval()
    
    # Process in batches
    num_windows = len(windows_faulty)
    
    with torch.no_grad():
        for batch_start in tqdm(range(0, num_windows, batch_size), desc="Creating graphs (batched)"):
            batch_end = min(batch_start + batch_size, num_windows)
            batch_windows = windows_faulty[batch_start:batch_end]
            batch_labels = labels_per_window[batch_start:batch_end]
            
            # Stack windows into batch tensor [batch_size, 512, 6]
            max_len = max(w.shape[0] for w in batch_windows)
            batch_tensor = torch.zeros(len(batch_windows), max_len, batch_windows[0].shape[1], 
                                     dtype=torch.float32, device=device)
            batch_mask = torch.zeros(len(batch_windows), max_len, batch_windows[0].shape[1],
                                   dtype=torch.float32, device=device)
            
            for i, window in enumerate(batch_windows):
                batch_tensor[i, :window.shape[0], :] = torch.tensor(window, dtype=torch.float32, device=device)
                batch_mask[i, :window.shape[0], :] = 1.0
            
            # Forward pass on batch
            outputs = tspulse_model(
                past_values=batch_tensor,
                past_observed_mask=batch_mask,
                return_loss=False
            )
            
            # Process each window in batch
            for i, (window, window_labels) in enumerate(zip(batch_windows, batch_labels)):
                # Keep reconstruction on device for processing
                reconstruction = outputs.reconstruction_outputs[0][i, :window.shape[0], :]
                
                # Convert to numpy only when needed (for compatibility)
                reconstruction_np = reconstruction.cpu().numpy()
                window_np = window
                
                timestep_graphs = create_timestep_graphs_vectorized(
                    window_np, reconstruction_np, window_labels, device=device
                )
                all_graphs.extend(timestep_graphs)
    
    return all_graphs

train_graphs = create_graphs_from_faulty_windows_batched(
    train_windows_faulty, 
    [train_point_labels[i*512:(i+1)*512] for i in range(len(train_windows_faulty))],
    tspulse_model, 
    device
)
val_graphs = create_graphs_from_faulty_windows_batched(
    val_windows_faulty,
    [val_point_labels[i*512:(i+1)*512] for i in range(len(val_windows_faulty))],
    tspulse_model,
    device
)
test_graphs = create_graphs_from_faulty_windows_batched(
    test_windows_faulty,
    [test_point_labels[i*512:(i+1)*512] for i in range(len(test_windows_faulty))],
    tspulse_model,
    device
)

Creating graphs:   0%|          | 0/17991 [00:00<?, ?it/s]/Users/darenpalmer/Desktop/UCL/CS/fyp.nosync/.venv/lib/python3.10/site-packages/numpy/lib/function_base.py:2897: RuntimeWarning: invalid value encountered in divide
  c /= stddev[:, None]
/Users/darenpalmer/Desktop/UCL/CS/fyp.nosync/.venv/lib/python3.10/site-packages/numpy/lib/function_base.py:2898: RuntimeWarning: invalid value encountered in divide
  c /= stddev[None, :]
Creating graphs:   3%|▎         | 450/17991 [02:59<1:56:37,  2.51it/s]


KeyboardInterrupt: 

In [ ]:
normal_sample = [g for g in val_graphs if g.y.item() == 0][:1000]
anomaly_sample = [g for g in val_graphs if g.y.item() == 1][:1000]

if len(normal_sample) > 0 and len(anomaly_sample) > 0:
    normal_feats = torch.stack([g.x.mean(dim=0) for g in normal_sample]).mean(dim=0)
    anomaly_feats = torch.stack([g.x.mean(dim=0) for g in anomaly_sample]).mean(dim=0)
    
    diff = (anomaly_feats - normal_feats).abs()
    
    if diff.max() > 0.5:
        print("✅ GOOD SEPARATION - Model should learn!")
    else:
        print("⚠️  Low separation - may need stronger faults")

Creating timestep graphs:   0%|          | 0/17991 [00:00<?, ?it/s]/Users/darenpalmer/Desktop/UCL/CS/fyp.nosync/.venv/lib/python3.10/site-packages/numpy/lib/function_base.py:2897: RuntimeWarning: invalid value encountered in divide
  c /= stddev[:, None]
/Users/darenpalmer/Desktop/UCL/CS/fyp.nosync/.venv/lib/python3.10/site-packages/numpy/lib/function_base.py:2898: RuntimeWarning: invalid value encountered in divide
  c /= stddev[None, :]
Creating timestep graphs:   0%|          | 18/4326 [00:06<27:15,  2.63it/s]/Users/darenpalmer/Desktop/UCL/CS/fyp.nosync/.venv/lib/python3.10/site-packages/numpy/lib/function_base.py:2897: RuntimeWarning: invalid value encountered in divide
  c /= stddev[:, None]
/Users/darenpalmer/Desktop/UCL/CS/fyp.nosync/.venv/lib/python3.10/site-packages/numpy/lib/function_base.py:2898: RuntimeWarning: invalid value encountered in divide
  c /= stddev[None, :]
Creating timestep graphs:   0%|          | 0/2462 [00:00<?, ?it/s]/Users/darenpalmer/Desktop/UCL/CS/fyp.no

In [ ]:
class GraphAttentionAnomalyDetector(nn.Module):
    def __init__(self, node_feature_dim=3, edge_feature_dim=1, hidden_dim=64, num_layers=3, num_heads=4, dropout=0.3):
        super().__init__()
        
        self.node_encoder = nn.Sequential(
            nn.Linear(node_feature_dim, hidden_dim),
            nn.BatchNorm1d(hidden_dim),
            nn.ReLU(),
            nn.Dropout(dropout)
        )
        
        self.gat_layers = nn.ModuleList()
        for i in range(num_layers):
            self.gat_layers.append(
                GATv2Conv(hidden_dim, hidden_dim // num_heads, heads=num_heads, dropout=dropout, edge_dim=edge_feature_dim)
            )
        
        self.classifier = nn.Sequential(
            nn.Linear(hidden_dim, 32),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(32, 1)
        )
    
    def forward(self, data):
        x, edge_index, edge_attr, batch = data.x, data.edge_index, data.edge_attr, data.batch
        
        x = self.node_encoder(x)
        
        for gat_layer in self.gat_layers:
            x = gat_layer(x, edge_index, edge_attr)
            x = F.elu(x)
        
        graph_embedding = torch.zeros(data.num_graphs, x.shape[1], device=x.device)
        for i in range(data.num_graphs):
            mask = (batch == i)
            graph_embedding[i] = x[mask].mean(dim=0)
        
        logits = self.classifier(graph_embedding).squeeze(-1)
        return logits

In [ ]:
anom_graphs = [g for g in train_graphs if g.y.item() == 1]
normal_graphs = [g for g in train_graphs if g.y.item() == 0]
repeat_factor = max(1, len(normal_graphs) // len(anom_graphs))
balanced_graphs = normal_graphs + (anom_graphs * repeat_factor)
random.shuffle(balanced_graphs)

train_graph_loader = GraphDataLoader(balanced_graphs, batch_size=32, shuffle=True)
val_graph_loader = GraphDataLoader(val_graphs, batch_size=32, shuffle=False)
test_graph_loader = GraphDataLoader(test_graphs, batch_size=32, shuffle=False)

In [ ]:
num_epochs = 50

gnn_model = GraphAttentionAnomalyDetector(
    node_feature_dim=10,
    edge_feature_dim=1,
    hidden_dim=64,
    num_layers=3,
    num_heads=4,
    dropout=0.4
).to(device)

with torch.no_grad():
    bias_init = -0.5
    gnn_model.classifier[-1].bias.fill_(bias_init)

pos_weight = torch.tensor([1.5]).to(device)
criterion = nn.BCEWithLogitsLoss(pos_weight=pos_weight)

optimizer = torch.optim.AdamW(gnn_model.parameters(), lr=1e-3, weight_decay=1e-3)
scheduler = torch.optim.lr_scheduler.OneCycleLR(
    optimizer, 
    max_lr=1e-3,
    epochs=num_epochs,
    steps_per_epoch=len(train_graph_loader),
    pct_start=0.1
)

In [ ]:
best_f1 = 0

for epoch in range(num_epochs):
    gnn_model.train()
    train_loss = 0
    train_preds, train_true = [], []
    
    for graph_batch in tqdm(train_graph_loader, desc=f"Epoch {epoch+1}"):
        graph_batch = graph_batch.to(device)
        labels = graph_batch.y
        
        optimizer.zero_grad()
        logits = gnn_model(graph_batch)
        loss = criterion(logits, labels)
        loss.backward()
        torch.nn.utils.clip_grad_norm_(gnn_model.parameters(), 1.0)
        optimizer.step()
        
        train_loss += loss.item()
        probs = torch.sigmoid(logits)
        train_preds.extend((probs > 0.5).cpu().numpy())
        train_true.extend(labels.cpu().numpy())
    
    train_f1 = f1_score(train_true, train_preds, zero_division=0)
    
    gnn_model.eval()
    val_loss = 0
    val_preds, val_true = [], []
    
    with torch.no_grad():
        for graph_batch in val_graph_loader:
            graph_batch = graph_batch.to(device)
            logits = gnn_model(graph_batch)
            loss = criterion(logits, graph_batch.y)
            
            val_loss += loss.item()
            probs = torch.sigmoid(logits)
            val_preds.extend((probs > 0.5).cpu().numpy())
            val_true.extend(graph_batch.y.cpu().numpy())
    
    val_f1 = f1_score(val_true, val_preds, zero_division=0)
    val_prec = precision_score(val_true, val_preds, zero_division=0)
    val_rec = recall_score(val_true, val_preds, zero_division=0)
    
    scheduler.step(val_loss)
    
    print(f"Epoch {epoch+1}/{num_epochs}: Train Loss={train_loss/len(train_graph_loader):.4f} F1={train_f1:.3f} | Val Loss={val_loss/len(val_graph_loader):.4f} P={val_prec:.3f} R={val_rec:.3f} F1={val_f1:.3f}")
    
    if val_f1 > best_f1:
        best_f1 = val_f1
        torch.save(gnn_model.state_dict(), 'gnn_sensor_failure_best.pt')

Epoch 1:   0%|          | 0/3376 [00:00<?, ?it/s]

Epoch 1: 100%|██████████| 3376/3376 [04:57<00:00, 11.35it/s]


Epoch 1/50: Train Loss=0.8583 F1=0.604 | Val Loss=0.8560 P=0.149 R=1.000 F1=0.260


Epoch 2: 100%|██████████| 3376/3376 [04:49<00:00, 11.64it/s]


Epoch 2/50: Train Loss=0.8501 F1=0.636 | Val Loss=0.8838 P=0.149 R=1.000 F1=0.260


...

Epoch 3/50: Train Loss=0.8481 F1=0.640 | Val Loss=0.8719 P=0.149 R=1.000 F1=0.260


Epoch 4: 100%|██████████| 3376/3376 [04:56<00:00, 11.37it/s]


Epoch 4/50: Train Loss=0.8480 F1=0.641 | Val Loss=0.8792 P=0.149 R=1.000 F1=0.260


Epoch 5:  76%|███████▌  | 2549/3376 [03:55<01:16, 10.84it/s]


KeyboardInterrupt: 

In [ ]:
gnn_model.eval()
sample_batch = next(iter(val_graph_loader)).to(device)

with torch.no_grad():
    logits = gnn_model(sample_batch)
    probs = torch.sigmoid(logits)

print(f"Logits: mean={logits.mean().item():.3f}, std={logits.std().item():.3f}")
print(f"Probs: mean={probs.mean().item():.3f}, std={probs.std().item():.3f}, range=[{probs.min().item():.3f}, {probs.max().item():.3f}]")



Diagnostic:
Logits mean: 0.327, std: 0.022
Probs mean: 0.581, std: 0.005
Probs range: [0.573, 0.594]
Probs > 0.5: 32/32

Sample probabilities: [0.5837079  0.5873156  0.58213586 0.581496   0.5825156  0.5819546
 0.5857725  0.5830759  0.58992773 0.59435415 0.59348726 0.58776927
 0.58491355 0.57658917 0.5725939  0.57354826 0.57609475 0.5780685
 0.5787998  0.57846594]

Feature check:
Node features mean: [-0.49025774 -0.42521778  0.27526927]
Node features std: [0.8027112  0.72955316 0.24922752]


In [ ]:
gnn_model.load_state_dict(torch.load('gnn_sensor_failure_best.pt'))
gnn_model.eval()

test_preds, test_true = [], []

with torch.no_grad():
    for graph_batch in tqdm(test_graph_loader, desc="Testing"):
        graph_batch = graph_batch.to(device)
        logits = gnn_model(graph_batch)
        probs = torch.sigmoid(logits)
        
        test_preds.extend((probs > 0.5).cpu().numpy())
        test_true.extend(graph_batch.y.cpu().numpy())

test_f1 = f1_score(test_true, test_preds, zero_division=0)
test_f2 = fbeta_score(test_true, test_preds, beta=2, zero_division=0)
test_prec = precision_score(test_true, test_preds, zero_division=0)
test_rec = recall_score(test_true, test_preds, zero_division=0)

print(f"Test Results: Precision={test_prec:.4f}, Recall={test_rec:.4f}, F1={test_f1:.4f}, F2={test_f2:.4f}")

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import confusion_matrix, roc_curve, auc, precision_recall_curve

def plot_confusion_matrix_pretty(y_true, y_pred, title="Confusion Matrix"):
    """Beautiful confusion matrix"""
    cm = confusion_matrix(y_true, y_pred)
    
    fig, ax = plt.subplots(figsize=(8, 6))
    
    # Normalize for percentages
    cm_norm = cm.astype('float') / cm.sum(axis=1)[:, np.newaxis] * 100
    
    # Plot
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', 
                cbar_kws={'label': 'Count'}, ax=ax, linewidths=2, linecolor='white')
    
    # Add percentages
    for i in range(2):
        for j in range(2):
            ax.text(j+0.5, i+0.7, f'({cm_norm[i, j]:.1f}%)', 
                   ha='center', va='center', fontsize=10, color='gray')
    
    ax.set_xlabel('Predicted Label', fontsize=12, fontweight='bold')
    ax.set_ylabel('True Label', fontsize=12, fontweight='bold')
    ax.set_title(title, fontsize=14, fontweight='bold', pad=15)
    ax.set_xticklabels(['Normal', 'Anomaly'], fontsize=11)
    ax.set_yticklabels(['Normal', 'Anomaly'], fontsize=11, rotation=0)
    
    # Add metrics text
    tn, fp, fn, tp = cm.ravel()
    metrics_text = f"TP: {tp}  |  TN: {tn}\nFP: {fp}  |  FN: {fn}"
    ax.text(1, -0.15, metrics_text, ha='center', fontsize=10, 
            transform=ax.transAxes, bbox=dict(boxstyle='round', facecolor='wheat', alpha=0.3))
    
    plt.tight_layout()
    plt.show()


def plot_roc_and_pr_curves(y_true, y_probs):
    """ROC and Precision-Recall curves"""
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))
    
    # ROC Curve
    fpr, tpr, _ = roc_curve(y_true, y_probs)
    roc_auc = auc(fpr, tpr)
    
    ax1.plot(fpr, tpr, 'b-', linewidth=2, label=f'ROC (AUC = {roc_auc:.3f})')
    ax1.plot([0, 1], [0, 1], 'r--', linewidth=2, label='Random Classifier')
    ax1.set_xlabel('False Positive Rate', fontsize=12)
    ax1.set_ylabel('True Positive Rate', fontsize=12)
    ax1.set_title('ROC Curve', fontsize=14, fontweight='bold')
    ax1.legend(fontsize=11)
    ax1.grid(True, alpha=0.3)
    ax1.set_xlim([0, 1])
    ax1.set_ylim([0, 1])
    
    # Precision-Recall Curve
    precision, recall, _ = precision_recall_curve(y_true, y_probs)
    pr_auc = auc(recall, precision)
    
    ax2.plot(recall, precision, 'g-', linewidth=2, label=f'PR (AUC = {pr_auc:.3f})')
    baseline = y_true.mean()
    ax2.plot([0, 1], [baseline, baseline], 'r--', linewidth=2, 
             label=f'Baseline ({baseline:.3f})')
    ax2.set_xlabel('Recall', fontsize=12)
    ax2.set_ylabel('Precision', fontsize=12)
    ax2.set_title('Precision-Recall Curve', fontsize=14, fontweight='bold')
    ax2.legend(fontsize=11)
    ax2.grid(True, alpha=0.3)
    ax2.set_xlim([0, 1])
    ax2.set_ylim([0, 1])
    
    plt.tight_layout()
    plt.show()


def plot_prediction_distribution(y_true, y_probs):
    """Distribution of prediction probabilities"""
    fig, ax = plt.subplots(figsize=(10, 6))
    
    # Separate by true class
    normal_probs = y_probs[y_true == 0]
    anomaly_probs = y_probs[y_true == 1]
    
    # Plot histograms
    ax.hist(normal_probs, bins=50, alpha=0.6, label='Normal', color='blue', density=True)
    ax.hist(anomaly_probs, bins=50, alpha=0.6, label='Anomaly', color='red', density=True)
    
    # Threshold line
    ax.axvline(0.5, color='green', linestyle='--', linewidth=2, label='Threshold (0.5)')
    
    ax.set_xlabel('Predicted Probability', fontsize=12)
    ax.set_ylabel('Density', fontsize=12)
    ax.set_title('Distribution of Prediction Probabilities', fontsize=14, fontweight='bold')
    ax.legend(fontsize=11)
    ax.grid(True, alpha=0.3, axis='y')
    
    plt.tight_layout()
    plt.show()


def plot_sensor_failure_examples(test_data_faulty, test_point_labels, test_graphs, 
                                 y_true, y_pred, y_probs, n_examples=4):
    """Visualize specific failure cases"""
    
    # Find interesting cases
    tp_idx = np.where((np.array(y_true) == 1) & (np.array(y_pred) == 1))[0]  # True positives
    fn_idx = np.where((np.array(y_true) == 1) & (np.array(y_pred) == 0))[0]  # False negatives
    fp_idx = np.where((np.array(y_true) == 0) & (np.array(y_pred) == 1))[0]  # False positives
    
    fig, axes = plt.subplots(n_examples, 1, figsize=(14, 3*n_examples))
    if n_examples == 1:
        axes = [axes]
    
    cases = [
        (tp_idx, "True Positive (Correctly Detected)", 'green'),
        (fn_idx, "False Negative (Missed Anomaly)", 'red'),
        (fp_idx, "False Positive (False Alarm)", 'orange'),
        (tp_idx, "True Positive (High Confidence)", 'darkgreen')
    ]
    
    for i, (indices, title, color) in enumerate(cases[:n_examples]):
        if len(indices) == 0:
            continue
            
        # Pick random example
        idx = np.random.choice(indices)
        graph = test_graphs[idx]
        
        # Extract sensor values from graph node features
        sensor_values = graph.x[:, 0].cpu().numpy()  # Mean values per sensor
        sensor_names = ['COOLANT', 'RPM', 'SPEED', 'THROTTLE', 'LOAD', 'MAP']
        
        ax = axes[i]
        bars = ax.bar(sensor_names, sensor_values, color=color, alpha=0.7, edgecolor='black')
        
        # Title with prediction info
        prob = y_probs[idx]
        ax.set_title(f"{title}\nPrediction: {prob:.3f} | Ground Truth: {'Anomaly' if y_true[idx]==1 else 'Normal'}", 
                    fontsize=12, fontweight='bold', color=color)
        ax.set_ylabel('Sensor Value', fontsize=11)
        ax.grid(True, alpha=0.3, axis='y')
        ax.tick_params(axis='x', rotation=45)
    
    plt.tight_layout()
    plt.show()

gnn_model.load_state_dict(torch.load('gnn_sensor_failure_best.pt'))
gnn_model.eval()

test_preds, test_true, test_probs_list = [], [], []

with torch.no_grad():
    for graph_batch in tqdm(test_graph_loader, desc="Evaluating"):
        graph_batch = graph_batch.to(device)
        logits = gnn_model(graph_batch)
        probs = torch.sigmoid(logits)
        
        test_probs_list.extend(probs.cpu().numpy())
        test_preds.extend((probs > 0.5).cpu().numpy())
        test_true.extend(graph_batch.y.cpu().numpy())

test_probs = np.array(test_probs_list)
test_preds = np.array(test_preds)
test_true = np.array(test_true)

test_f1 = f1_score(test_true, test_preds, zero_division=0)
test_f2 = fbeta_score(test_true, test_preds, beta=2, zero_division=0)
test_prec = precision_score(test_true, test_preds, zero_division=0)
test_rec = recall_score(test_true, test_preds, zero_division=0)

print(f"Test Results: Precision={test_prec:.4f}, Recall={test_rec:.4f}, F1={test_f1:.4f}, F2={test_f2:.4f}")

plot_confusion_matrix_pretty(test_true, test_preds, "Test Set Confusion Matrix")
plot_roc_and_pr_curves(test_true, test_probs)
plot_prediction_distribution(test_true, test_probs)
plot_sensor_failure_examples(test_data_faulty, test_point_labels, test_graphs, test_true, test_preds, test_probs, n_examples=4)
